# Question 4.1

We transform each patient's time series data to a simple aggregated piece of text. To save valuable text and token space and run models faster, we came up with a
compact summary. We tried many different approaches but what we found works best is simply taking the last value for each patient. However, this might be due to the context window of these local LLMs being quite small so some of the data might be lost.

In [2]:
#Load the data
import numpy as np
import pandas as pd

df_train = pd.read_parquet("set_a.parquet")
df_val   = pd.read_parquet("set_b.parquet")
df_test  = pd.read_parquet("set_c.parquet")

#note same function as in Exercise 2.1 to get the simple extracted features
DYNAMIC_VARS = sorted([
    'ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP',
    'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2',
    'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH'
])
STATIC_VARS = ['Age', 'Gender', 'Height', 'StaticWeight']
ALL_FEATURES = DYNAMIC_VARS + STATIC_VARS  # 41 features total

def extract_simple_features(df):
    """
    Per patient:
      - Forward-fill measurements over time to ignore gaps/NaNs.
      - Dynamic variables: take the last non-null value after ffill.
      - Static variables: take as-is (they are constant).
    Returns one row per patient with features + label.
    """
    df_sorted = df.sort_values(["RecordID", "hour"])
    
    # Forward-fill all columns within each patient group
    df_filled = df_sorted.groupby("RecordID").ffill()
    
    # groupby.ffill() drops the grouping key, so we map it back from the sorted dataframe
    df_filled["RecordID"] = df_sorted["RecordID"] 
    
    # Drop duplicates to keep only the final chronological row per patient
    last_row = df_filled.drop_duplicates(subset=["RecordID"], keep="last")

    features = last_row[["RecordID"] + ALL_FEATURES].copy()
    labels   = last_row[["RecordID", "In-hospital_death"]].copy()
    
    return features, labels

X_train, y_train = extract_simple_features(df_train)
X_val,   y_val   = extract_simple_features(df_val)
X_test,  y_test  = extract_simple_features(df_test)


In [3]:
# Short-name to full-name and unit mappings for all available variables
FEATURE_FULL_NAMES = {
    "RecordID": "ICU stay identifier",
    "hour": "Hours since ICU admission",
    "ALP": "Alkaline phosphatase",
    "ALT": "Alanine transaminase",
    "AST": "Aspartate transaminase",
    "Albumin": "Serum albumin",
    "BUN": "Blood urea nitrogen",
    "Bilirubin": "Total bilirubin",
    "Cholesterol": "Total cholesterol",
    "Creatinine": "Serum creatinine",
    "DiasABP": "Invasive diastolic arterial blood pressure",
    "FiO2": "Fraction of inspired oxygen",
    "GCS": "Glasgow Coma Score",
    "Glucose": "Serum glucose",
    "HCO3": "Serum bicarbonate",
    "HCT": "Hematocrit",
    "HR": "Heart rate",
    "K": "Serum potassium",
    "Lactate": "Serum lactate",
    "MAP": "Invasive mean arterial blood pressure",
    "MechVent": "Mechanical ventilation flag",
    "Mg": "Serum magnesium",
    "NIDiasABP": "Non-invasive diastolic arterial blood pressure",
    "NIMAP": "Non-invasive mean arterial blood pressure",
    "NISysABP": "Non-invasive systolic arterial blood pressure",
    "Na": "Serum sodium",
    "PaCO2": "Partial pressure of arterial CO2",
    "PaO2": "Partial pressure of arterial O2",
    "Platelets": "Platelet count",
    "RespRate": "Respiration rate",
    "SaO2": "Oxygen saturation of hemoglobin",
    "SysABP": "Invasive systolic arterial blood pressure",
    "Temp": "Body temperature",
    "TroponinI": "Troponin I",
    "TroponinT": "Troponin T",
    "Urine": "Urine output",
    "WBC": "White blood cell count",
    "Weight": "Observed weight",
    "pH": "Arterial pH",
    "Age": "Age",
    "Gender": "Gender (0=female, 1=male)",
    "Height": "Height",
    "StaticWeight": "Admission weight (static)",
    "ICUType": "ICU type (1=CCU, 2=CSRU, 3=MICU, 4=SICU)",
    "In-hospital_death": "In-hospital death outcome"
}

FEATURE_UNITS = {
    "RecordID": "none",
    "hour": "hour",
    "ALP": "IU/L",
    "ALT": "IU/L",
    "AST": "IU/L",
    "Albumin": "g/dL",
    "BUN": "mg/dL",
    "Bilirubin": "mg/dL",
    "Cholesterol": "mg/dL",
    "Creatinine": "mg/dL",
    "DiasABP": "mmHg",
    "FiO2": "fraction (0-1)",
    "GCS": "score (3-15)",
    "Glucose": "mg/dL",
    "HCO3": "mmol/L",
    "HCT": "percent",
    "HR": "bpm",
    "K": "mEq/L",
    "Lactate": "mmol/L",
    "MAP": "mmHg",
    "MechVent": "binary (0/1)",
    "Mg": "mmol/L",
    "NIDiasABP": "mmHg",
    "NIMAP": "mmHg",
    "NISysABP": "mmHg",
    "Na": "mEq/L",
    "PaCO2": "mmHg",
    "PaO2": "mmHg",
    "Platelets": "cells/nL",
    "RespRate": "bpm",
    "SaO2": "percent",
    "SysABP": "mmHg",
    "Temp": "degC",
    "TroponinI": "ug/L",
    "TroponinT": "ug/L",
    "Urine": "mL",
    "WBC": "cells/nL",
    "Weight": "kg",
    "pH": "pH units",
    "Age": "years",
    "Gender": "binary (0/1)",
    "Height": "cm",
    "StaticWeight": "kg",
    "ICUType": "categorical (1-4)",
    "In-hospital_death": "binary (0/1)"
}

The function below creates an LLM readable summary of a row of data. Note here that a row data already contains the feature for each variable. So it simply puts it in to text and adds the units to it.

In [4]:
# craete the summary for the LLM to predict whether the patient is alive or not

def create_summary(row):
    summary = []
    for feature in ALL_FEATURES:
        value = row[feature]
        if pd.isna(value):
            continue  # Skip features with missing values
        summary.append(f"last known {FEATURE_FULL_NAMES[feature]} value: {value} {FEATURE_UNITS[feature]}.")
    return "; ".join(summary)

## Predicting a number 1 or 0

In [4]:
from ollama import chat
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

predictions = []
max_patients = 5000
model_name = "gemma2:2b"

# get positive and negative samples
pos_idx = y_train.index[y_train["In-hospital_death"] == 1]
neg_idx = y_train.index[y_train["In-hospital_death"] == 0]

n_pos = 1
n_neg = 3

few_shot_examples = pd.concat(
    [X_train.loc[pos_idx].head(n_pos), X_train.loc[neg_idx].head(n_neg)],
    axis=0
).reset_index(drop=True)

few_shot_answers = pd.concat(
    [y_train.loc[pos_idx].head(n_pos), y_train.loc[neg_idx].head(n_neg)],
    axis=0
)["In-hospital_death"].reset_index(drop=True)

few_shot_prompt = "Your task is to predict ICU patients' in-hospital death label (0 or 1).\n"
few_shot_prompt += "You are given each patient's last measured values before discharge (0) or death (1).\n"
for i in range(len(few_shot_examples)):
    label = int(few_shot_answers[i])
    few_shot_prompt += f"Example {i+1}: {create_summary(few_shot_examples.iloc[i])} -> {label}\n"

few_shot_prompt += (
    "Based on the patient's last known measurements, predict in-hospital death.\n"
    "Output exactly one character: 0 or 1 only (no extra text).\n"
)

effective_n = min(max_patients, len(X_test), len(y_test))
parse_fallbacks = 0

# Inference on test
for num in tqdm(range(effective_n), desc="Scoring test patients"):
    prompt = (
        few_shot_prompt
        + "Now, predict this patient (0 or 1):\n"
        + create_summary(X_test.iloc[num])
        + "\nAnswer:"
    )

    response = chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.1},
    )

    text = response.message.content.strip()

    # Strict parse first, then fallback parse
    if text in {"0", "1"}:
        pred = int(text)
    else:
        m = re.search(r"\b[01]\b", text)
        if m is None:
            pred = 0   # fallback to majority class
            parse_fallbacks += 1
        else:
            pred = int(m.group())

    predictions.append(float(pred))

# Evaluate on the test
y_true = y_test.iloc[:effective_n]["In-hospital_death"].to_numpy()

print("Split used: test")
print(f"Patients scored: {effective_n}")
print(f"Parse fallbacks to 0: {parse_fallbacks}")
print(f"Average predicted mortality label: {np.mean(predictions):.3f}")
print(f"True mortality rate in evaluated test slice: {np.mean(y_true):.3f}")

# AUROC needs both classes present
if np.unique(y_true).size < 2:
    print("AuROC undefined for this slice: only one class present in y_true.")
else:
    auroc = roc_auc_score(y_true, predictions)
    print(f"AuROC: {auroc:.3f}")

# AUPRC requires at least one positive
if np.sum(y_true == 1) == 0:
    print("AuPRC undefined for this slice: no positive cases in y_true.")
else:
    auprc = average_precision_score(y_true, predictions)
    print(f"AuPRC: {auprc:.3f}")

/cluster/courses/ml4h/jupyter/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Scoring test patients: 100%|██████████| 4000/4000 [26:53<00:00,  2.48it/s]

Split used: test
Patients scored: 4000
Parse fallbacks to 0: 0
Average predicted mortality label: 0.032
True mortality rate in evaluated test slice: 0.146
AuROC: 0.521
AuPRC: 0.158


## Predicting a Number between 0 and 10

In [4]:
from ollama import chat
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

predictions = []   # probabilities in [0,1], used for metrics
raw_scores = []    # raw model scores in [0,10]
max_patients = 5000
model_name = "gemma2:2b"

# get positive and negative samples
pos_idx = y_train.index[y_train["In-hospital_death"] == 1]
neg_idx = y_train.index[y_train["In-hospital_death"] == 0]

n_pos = 1
n_neg = 3

few_shot_examples = pd.concat(
    [X_train.loc[pos_idx].head(n_pos), X_train.loc[neg_idx].head(n_neg)],
    axis=0
).reset_index(drop=True)

few_shot_answers = pd.concat(
    [y_train.loc[pos_idx].head(n_pos), y_train.loc[neg_idx].head(n_neg)],
    axis=0
)["In-hospital_death"].reset_index(drop=True)

# Prompt style aligned with your probability version, but with 0-10 output scale
few_shot_prompt = "Your task is to predict ICU patients' in-hospital death risk score (0 to 10).\n"
few_shot_prompt += "You are given each patient's last measured values before being discharged (0) or dying (1).\n"
for i in range(len(few_shot_examples)):
    score_label = 10.0 if few_shot_answers[i] == 1 else 0.0
    few_shot_prompt += f"Example {i+1}: {create_summary(few_shot_examples.iloc[i])} -> {score_label}\n"

few_shot_prompt += (
    "Based on the patient's last known measurements, predict the in-hospital death risk score.\n"
    "Output one number between 0 and 10 only (decimals allowed, no extra text).\n"
)

effective_n = min(max_patients, len(X_test), len(y_test))
parse_fallbacks = 0

# Inference on test
for num in tqdm(range(effective_n), desc="Scoring test patients"):
    prompt = (
        few_shot_prompt
        + "Now, predict this patient (risk score 0-10):\n"
        + create_summary(X_test.iloc[num])
        + "\nAnswer:"
    )

    response = chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.1},
    )

    text = response.message.content.strip()
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", text)

    if m is None:
        score = 5.0
        parse_fallbacks += 1
    else:
        score = float(m.group())

    score = float(np.clip(score, 0.0, 10.0))
    prob = score / 10.0

    raw_scores.append(score)
    predictions.append(prob)

# Evaluate on test
y_true = y_test.iloc[:effective_n]["In-hospital_death"].to_numpy()

print("Split used: test")
print(f"Patients scored: {effective_n}")
print(f"Parse fallbacks to 5.0: {parse_fallbacks}")
print(f"Average predicted raw score (0-10): {np.mean(raw_scores):.3f}")
print(f"Average predicted mortality probability: {np.mean(predictions):.3f}")
print(f"True mortality rate in evaluated test slice: {np.mean(y_true):.3f}")

# AUROC needs both classes present
if np.unique(y_true).size < 2:
    print("AuROC undefined for this slice: only one class present in y_true.")
else:
    auroc = roc_auc_score(y_true, predictions)
    print(f"AuROC: {auroc:.3f}")

# AUPRC requires at least one positive
if np.sum(y_true == 1) == 0:
    print("AuPRC undefined for this slice: no positive cases in y_true.")
else:
    auprc = average_precision_score(y_true, predictions)
    print(f"AuPRC: {auprc:.3f}")

/cluster/courses/ml4h/jupyter/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Scoring test patients: 100%|██████████| 4000/4000 [43:40<00:00,  1.53it/s]  

Split used: test
Patients scored: 4000
Parse fallbacks to 5.0: 1
Average predicted raw score (0-10): 1.214
Average predicted mortality probability: 0.121
True mortality rate in evaluated test slice: 0.146
AuROC: 0.541
AuPRC: 0.162


## Predict Probability Directly

In [5]:
from ollama import chat
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

predictions = []
max_patients = 5000
model_name = "gemma2:2b"

# get positive and negative samples
pos_idx = y_train.index[y_train["In-hospital_death"] == 1]
neg_idx = y_train.index[y_train["In-hospital_death"] == 0]

n_pos = 1
n_neg = 3

few_shot_examples = pd.concat(
    [X_train.loc[pos_idx].head(n_pos), X_train.loc[neg_idx].head(n_neg)],
    axis=0).reset_index(drop=True)

few_shot_answers = pd.concat(
    [y_train.loc[pos_idx].head(n_pos), y_train.loc[neg_idx].head(n_neg)],
    axis=0)["In-hospital_death"].reset_index(drop=True)

few_shot_prompt = "Your task is to predict ICU patients' and their probability of in-hospital death. (0 to 1):\n"
few_shot_prompt += "You are given each patient's last measured values before being discharged 0 or dying 1.\n"
for i in range(len(few_shot_examples)):
    prob_label = 1.0 if few_shot_answers[i] == 1 else 0.0
    few_shot_prompt += f"Example {i+1}: {create_summary(few_shot_examples.iloc[i])} -> {prob_label}\n"

few_shot_prompt += (
    "Based on the patient's last known measurements, predict the probability of in-hospital death.\n"
    "Output one number between 0 and 1 only (decimals allowed, no extra text).\n"
)

effective_n = min(max_patients, len(X_test), len(y_test))
parse_fallbacks = 0

# Inference on test
for num in tqdm(range(effective_n), desc="Scoring test patients"):
    prompt = (
        few_shot_prompt
        + "Now, predict this patient (probability 0-1):\n"
        + create_summary(X_test.iloc[num])
        + "\nAnswer:"
    )

    response = chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.1},
    )

    text = response.message.content.strip()
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", text)

    if m is None:
        prob = 0.5
        parse_fallbacks += 1
    else:
        prob = float(m.group())

    prob = float(np.clip(prob, 0.0, 1.0))
    predictions.append(prob)

# Evaluate on the SAME test slice
y_true = y_test.iloc[:effective_n]["In-hospital_death"].to_numpy()

print(f"Split used: test")
print(f"Patients scored: {effective_n}")
print(f"Parse fallbacks to 0.5: {parse_fallbacks}")
print(f"Average predicted mortality probability: {np.mean(predictions):.3f}")
print(f"True mortality rate in evaluated test slice: {np.mean(y_true):.3f}")

# AUROC needs both classes present
if np.unique(y_true).size < 2:
    print("AuROC undefined for this slice: only one class present in y_true.")
else:
    auroc = roc_auc_score(y_true, predictions)
    print(f"AuROC: {auroc:.3f}")

# AUPRC requires at least one positive
if np.sum(y_true == 1) == 0:
    print("AuPRC undefined for this slice: no positive cases in y_true.")
else:
    auprc = average_precision_score(y_true, predictions)
    print(f"AuPRC: {auprc:.3f}")

/cluster/courses/ml4h/jupyter/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Scoring test patients: 100%|██████████| 4000/4000 [29:05<00:00,  2.29it/s]

Split used: test
Patients scored: 4000
Parse fallbacks to 0.5: 1
Average predicted mortality probability: 0.111
True mortality rate in evaluated test slice: 0.146
AuROC: 0.559
AuPRC: 0.198
